<a href="https://colab.research.google.com/github/BrunaFerreira/Mestrado_UNIFESP/blob/main/Revisao_3_Perguntas_Pesquisa_Lista_Artigos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Base Completa de Artigos**

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
path = '/content/drive/MyDrive/0_Mestrado_unifesp/3_Pesquisa/Revisao/202607_Revisao_Completa/2_Perguntas_Pesquisa/'
path_to = '/content/drive/MyDrive/0_Mestrado_unifesp/3_Pesquisa/Revisao/202607_Revisao_Completa/1_Artigos_Filtrados/'

df_antigo_perguntas = pd.read_csv(path + 'VF_Perguntas_Respostas.csv')
df_atual_perguntas = pd.read_csv(path_to + '4_Lista_Completa_Artigos_Abstracts_202607.csv')

## **Perguntas e Respostas Revisao Anterior : Ajustes**

In [3]:
perguntas = df_antigo_perguntas.columns.tolist()
df_antigo_perguntas = df_antigo_perguntas.copy()

# Usar a primeira linha como cabeçalho
df_antigo_perguntas.columns = df_antigo_perguntas.iloc[0]
# Remover a primeira linha e reorganizar o índice
df_antigo_perguntas = df_antigo_perguntas.iloc[1:].reset_index(drop=True)
# Retirar o nome do eixo das colunas
df_antigo_perguntas.columns.name = None
mascara = (
    df_antigo_perguntas["Questao 1"].isna()
    | df_antigo_perguntas["Questao 1"].astype("string").str.strip().eq("")
)

df_antigo_perguntas.loc[mascara, "Questao 1"] = df_antigo_perguntas.loc[mascara, "Nome"]
df_antigo_perguntas.drop(columns=["Questao 9"], inplace=True)
df_antigo_perguntas["Status"] = (
    df_antigo_perguntas["Status"] != "Nao da pra baixar arquivo"
).astype(int)
df_antigo_perguntas.columns = ['DOI', 'Codigo', 'Query', 'Base', 'Status_Acesso', 'PDF', 'Num Reference 2025',
       'Nome', 'Questao_1', 'Questao_18', 'Questao_2', 'Questao_3',
       'Questao_4', 'Questao_5', 'Questao_6', 'Questao_7', 'Questao_8',
        'Questao_10', 'Questao_11', 'Questao_12', 'Questao_13',
       'Questao_14', 'Questao_15', 'Questao_16', 'Questao_17']

In [4]:
df_antigo_perguntas.Status_Acesso.value_counts()

,count
Status_Acesso,
1,154
0,5


In [5]:
df_antigo_perguntas = df_antigo_perguntas[['DOI', 'Query','Nome','Status_Acesso','Questao_1', 'Questao_18', 'Questao_2', 'Questao_3',
       'Questao_4', 'Questao_5', 'Questao_6', 'Questao_7', 'Questao_8',
        'Questao_10', 'Questao_11', 'Questao_12', 'Questao_13',
       'Questao_14', 'Questao_15', 'Questao_16', 'Questao_17']]

key = ['Questao_1', 'Questao_18', 'Questao_2', 'Questao_3',
       'Questao_4', 'Questao_5', 'Questao_6', 'Questao_7', 'Questao_8',
        'Questao_10', 'Questao_11', 'Questao_12', 'Questao_13',
       'Questao_14', 'Questao_15', 'Questao_16', 'Questao_17']
valor = [item for item in perguntas if not str(item).startswith("Unnamed:")]
perguntas = dict(zip(key, valor))

df_antigo_perguntas["_query_join"] = pd.to_numeric(df_antigo_perguntas["Query"], errors="coerce").astype("Int64")
df_antigo_perguntas["_nome_join"] = df_antigo_perguntas["Nome"].astype("string").str[:30]

df_antigo_perguntas.shape

(159, 23)

## **Base de Artigos Completa**

### Artigos da base anterior que nao tem acesso completo

In [6]:
restritos = [
    "10.1002/9781394358212.ch33",
    "10.1007/978-981-95-9027-8_7",
    "10.1117/12.3109369",
]

restritos += (
    df_antigo_perguntas.loc[
        df_antigo_perguntas["Status_Acesso"] == 0,
        "DOI"
    ]
    .dropna()
    .tolist()
)
restritos = list(dict.fromkeys(restritos))

In [7]:
df_atual_perguntas.loc[df_atual_perguntas["DOI"].isin(restritos), "Status_Acesso"] = 0
df_atual_perguntas.loc[df_atual_perguntas["DOI"].isin(restritos), "Status_Final"] = 0
df_atual_perguntas["Status_Acesso"] = df_atual_perguntas["Status_Acesso"].fillna(1)

In [8]:
df_atual_perguntas.Status_Acesso.value_counts()

,count
Status_Acesso,
1.0,223
0.0,8


### **Filtro de Artigos com abstracts selecionados**

---



In [9]:
df_atual_perguntas = df_atual_perguntas[df_atual_perguntas['Status_Final']==1]

In [10]:
df_atual_perguntas.shape

(161, 10)

## **Inclusão de Respostas das 15 perguntas para os artigos**

In [11]:
df_atual_perguntas = df_atual_perguntas[['DOI',  'Query', 'Base','Nome','Criterio_Final','Status_Final'
]]
df_atual_perguntas["_query_join"] = pd.to_numeric(df_atual_perguntas["Query"], errors="coerce").astype("Int64")
df_atual_perguntas["_nome_join"] = df_atual_perguntas["Nome"].astype("string").str[:30]

df_atual_perguntas.head()

,DOI,Query,Base,Nome,Criterio_Final,Status_Final,_query_join,_nome_join
1,10.1007/978-981-97-7679-5_1,6.0,Scopus,CLASSIFICATION OF GOUGEROT SJOGREN SYNDROME BA...,Parecido com nosso objetivo,1.0,6,CLASSIFICATION OF GOUGEROT SJO
3,10.1016/j.jtos.2024.08.002,6.0,Scopus,DEEP LEARNING BASED ANALYSIS OF IN VIVO CONFOC...,NaN,1.0,6,DEEP LEARNING BASED ANALYSIS O
4,10.1038/s41598-023-42719-5,6.0,Scopus,APPLICATION OF SERUM SERS TECHNOLOGY COMBINED ...,NaN,1.0,6,APPLICATION OF SERUM SERS TECH
6,10.1016/j.bspc.2026.109615,6.0,Scopus,DIAGNOSTIC PERFORMANCE OF DEEP LEARNING MODELS...,Parecido com nosso objetivo,1.0,6,DIAGNOSTIC PERFORMANCE OF DEEP
7,10.3390/diagnostics13142373,6.0,Scopus,DETECTION OF HYDROXYCHLOROQUINE RETINOPATHY VI...,NaN,1.0,6,DETECTION OF HYDROXYCHLOROQUIN


In [12]:
# LEFT JOIN
df = df_atual_perguntas.merge(
    df_antigo_perguntas,
    how="left",
    left_on=["DOI", "_query_join", "_nome_join"],
    right_on=["DOI", "_query_join", "_nome_join"],
    suffixes=("_A", "_B")
)
# Remover colunas auxiliares
df = df.drop(
    columns=["_query_join", "_nome_join",'Nome_B','Status_Acesso','Query_B']
)
df.head(5)

,DOI,Query_A,Base,Nome_A,Criterio_Final,Status_Final,Questao_1,Questao_18,Questao_2,Questao_3,...,Questao_7,Questao_8,Questao_10,Questao_11,Questao_12,Questao_13,Questao_14,Questao_15,Questao_16,Questao_17
0,10.1007/978-981-97-7679-5_1,6.0,Scopus,CLASSIFICATION OF GOUGEROT SJOGREN SYNDROME BA...,Parecido com nosso objetivo,1.0,CLASSIFICATION OF GOUGEROT SJOGREN SYNDROME BA...,Não,Sem TL,"U-Net (adaptado), FCN e VGG (encoder)",...,Pacientes com síndrome de Gougerot-Sjögren\nPa...,Sem atributos sensiveis,Publico e Privado,Não especificado,Imagens de ultrassonografia (US) das glândulas...,Classificação e segmentação,Não mencionado,Não mencionado,Dice coefficient\nIntersection over Union (IoU...,Variabilidade na qualidade e aquisição das ima...
1,10.1016/j.jtos.2024.08.002,6.0,Scopus,DEEP LEARNING BASED ANALYSIS OF IN VIVO CONFOC...,NaN,1.0,Deep-learning based analysis of in-vivo confoc...,Não,Não mencionado,SNP-Net,...,3,"Sim (idade, genero, condições sistemicas",Privados,Pacientes: 104.\r\nOlhos analisados: 160.\r\nD...,imagens de microscopia confocal in vivo (IVCM,Segmentação,Não mencionado,Representatividade Geográfica\r\nDistribuição ...,Métricas de Segmentação (Dice Similarity Coeff...,Qualidade e Variabilidade das Imagens\r\nTama...
2,10.1038/s41598-023-42719-5,6.0,Scopus,APPLICATION OF SERUM SERS TECHNOLOGY COMBINED ...,NaN,1.0,Application of serum SERS technology combined ...,Não,Não mencionado,AlexNet\r\nResNet (Residual Network)\r\nSqueez...,...,2,Sim(idade e genero),Privados,pSS: 27 espectros.\r\nDN: 30 espectros.\r\nHC:...,espectroscopia Raman aprimorada por superfície...,Classificação,Não mencionado,1. Viés de tamanho da amostra\r\n2. Viés demog...,\r\nAcurácia (Accuracy)\r\nSensibilidade (Sens...,1. Tamanho limitado da amostra\r\n2. Variabili...
3,10.1016/j.bspc.2026.109615,6.0,Scopus,DIAGNOSTIC PERFORMANCE OF DEEP LEARNING MODELS...,Parecido com nosso objetivo,1.0,Diagnostic performance of deep learning models...,NaN,ImageNet,YOLO11n-cls\nYOLO11s-cls\nYOLO11m-cls\nYOLO11l...,...,A variável resposta possui 2 classes (classifi...,Sem atributos sensiveis,Privado,Não especificado,Ultrassonografia (US) das glândulas parótidas,Classificação,Não foi utilizada abordagem de fairness,Não especificado,Acurácia\nSensibilidade\nEspecificidade\nAUC,Dificuldade no diagnóstico precoce da Síndrome...
4,10.3390/diagnostics13142373,6.0,Scopus,DETECTION OF HYDROXYCHLOROQUINE RETINOPATHY VI...,NaN,1.0,Detection of Hydroxychloroquine Retinopathy vi...,Não,ImageNet,- ResNet (Residual Network)\r\n- VGG (Visual ...,...,2,Não,Privados,Não mencionado,imagens hiperespectrais oftalmoscópicas,classificação,Não mencionado,Viés de amostra: Falta de diversidade demográf...,\r\nAcurácia (Accuracy)\r\nPrecisão (Precision...,1. Tamanho limitado e diversidade dos dados.\r...


## **Filtro de artigos a serem feitas Perguntas de pesquisa**

In [13]:
df_novos = df[df['Questao_1'].isna()]

In [14]:
df_novos["Nome_20"] = df_novos["Nome_A"].astype("string").str[:20]

/tmp/ipykernel_47666/3751039615.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_novos["Nome_20"] = df_novos["Nome_A"].astype("string").str[:20]


In [15]:
df_novos.shape

(7, 24)

In [16]:
df_novos[['Nome_A','DOI','Nome_20']]

,Nome_A,DOI,Nome_20
28,TRANSFERRING HEALTHCARE RISK PREDICTION MODELS...,10.1038/s44401-026-00097-w,TRANSFERRING HEALTHC
106,HOW IS BIAS LEARNED IN MEDICAL IMAGE ANALYSIS ...,10.1007/s10278-026-02073-0,HOW IS BIAS LEARNED
121,DERMAVIGNET: A HYBRID VISION TRANSFORMER AND C...,10.1109/ICCIT68739.2025.11490111,DERMAVIGNET: A HYBRI
133,FAIRVLM: ENHANCING FAIRNESS AND PROMPT SENSITI...,10.1109/WACV61042.2026.00719,FAIRVLM: ENHANCING F
157,EQUITABLE HEALTH INTELLIGENCE: AN OPEN BENCHMA...,10.64898/2026.05.29.728755,EQUITABLE HEALTH INT
159,FAIRGEN: PREFERENCE ALIGNED DIFFUSION FOR DEMO...,10.1038/s41746-026-02868-z,FAIRGEN: PREFERENCE
160,ETHICAL IMPLICATIONS OF THE USE OF AI BASED TE...,10.3310/GJSE4912,ETHICAL IMPLICATIONS


In [17]:
df_novos.to_csv(path + '1_Artigos_Novos_Para_Perguntas.csv', index=False)

In [18]:
perguntas

{'Questao_1': 'Qual o titulo do artigo?',
 'Questao_18': 'Retorne "Sim" se for um artigo de revisão, caso contrario retorne Não',
 'Questao_2': 'Se o artigo citar o uso de transfer learning, Cite apenas o nome onde as redes foram pre treinadas para fazer Transfer Learning citadas no artigo. Caso contrario retorne "Sem TL"',
 'Questao_3': 'Retorne o nome do modelos de redes neurais ajustados para realizar a tarefa, citados no artigo',
 'Questao_4': 'Se o artigo utiliza modelos de redes neurais ja existentes retorne Existentes caso contrario retorne o nome da nova rede proposta',
 'Questao_5': 'Retorne apenas o nome da doença analisada no artigo',
 'Questao_6': 'Retorne uma lista sem repetições dos dados utilizados pelo artigo',
 'Questao_7': 'Retorne quantas classes tem a variavel resposta e quais sao',
 'Questao_8': 'Retone a lista de atributos pessoais que a base de dados contem, caso nao tenha retorne apenas "Sem atributos sensiveis"',
 'Questao_10': 'Retorne "Publico" se dados utili

## **Coleta de respostas**

Etapa feita no Codex.

In [19]:
df_respostas = pd.read_csv(path + '2_Lista_Artigos_Perguntas_Respostas_GPT.csv')

In [20]:
df_respostas.columns

Index(['DOI', 'Codigo_A', 'Query_A', 'Base_A', 'Artigo', 'Nome_A', 'Codigo_B',
       'Query_B', 'Base_B', 'Status', 'Nome_B', 'Questao_1', 'Questao_18',
       'Questao_2', 'Questao_3', 'Questao_4', 'Questao_5', 'Questao_6',
       'Questao_7', 'Questao_8', 'Questao_10', 'Questao_11', 'Questao_12',
       'Questao_13', 'Questao_14', 'Questao_15', 'Questao_16', 'Questao_17',
       'Nome_20'],
      dtype='object')

In [21]:
df_respostas  = df_respostas[['DOI', 'Questao_1', 'Questao_18',
       'Questao_2', 'Questao_3', 'Questao_4', 'Questao_5', 'Questao_6',
       'Questao_7', 'Questao_8', 'Questao_10', 'Questao_11', 'Questao_12',
       'Questao_13', 'Questao_14', 'Questao_15', 'Questao_16', 'Questao_17']]

In [22]:
df_respostas.head(15)

,DOI,Questao_1,Questao_18,Questao_2,Questao_3,Questao_4,Questao_5,Questao_6,Questao_7,Questao_8,Questao_10,Questao_11,Questao_12,Questao_13,Questao_14,Questao_15,Questao_16,Questao_17
0,10.1038/s44401-026-00097-w,Transferring healthcare risk prediction models...,Não,População Medicaid do estado de Washington (do...,Regressão logística source-only; regressão log...,Existentes,Utilização aguda de cuidados de saúde (visita ...,Arquivos desidentificados de claims e inscriçã...,2 classes: ocorrência de qualquer visita ao pr...,Idade; gênero; raça/etnia,Privado,49.645 participantes: Washington n=20.744 e Vi...,"Dados tabulares longitudinais de claims, inscr...",Classificação,Avaliação por equalized odds difference em raç...,"Domain shift demográfico, clínico, regulatório...",AUC/AUROC; Youden's J; sensibilidade/TPR; espe...,Generalização limitada a apenas dois estados; ...
1,10.1007/s10278-026-02073-0,How is Bias Learned in Medical Image Analysis ...,Não,ImageNet,DenseNet-121; classificadores lineares de sond...,Existentes,Anormalidades torácicas em radiografias de tórax,CheXpert; MIMIC-CXR,Classificação multirrótulo com 14 achados: No ...,"Idade; sexo; raça (White, Black/African Americ...",Publico,Mais de 590.000 imagens: CheXpert com mais de ...,Radiografias de tórax; relatórios radiológicos...,Classificação multirrótulo,"Auditoria estratificada por idade, raça e sexo...",Sub-representação racial; exclusão de raças au...,AUC/AUROC; TPR; FPR; disparidade de TPR; confi...,Necessidade de validação em domínios de imagem...
2,10.1109/ICCIT68739.2025.11490111,DermaViGNet: A Hybrid Vision Transformer and C...,Não,Não especificado no artigo; o VGG16 é descrito...,DermaViGNet; VGG19; VGG16; VGG18; MobileNet; D...,DermaViGNet,Acne; vitiligo; hiperpigmentação; psoríase ung...,Skin Disease Classification Dataset com imagen...,5 classes: Acne; Vitiligo; Hyperpigmentation; ...,Sem atributos sensiveis,Publico e Privado,9.548 imagens: Acne 1.148; Vitiligo 2.016; Hip...,Imagens dermatoscópicas,Classificação,Inclusão de doenças comuns e sub-representadas...,"Desbalanceamento entre doenças, com condições ...",Acurácia; precisão; recall; F1-score; loss; ma...,Classificar condições raras e sub-representada...
3,10.1109/WACV61042.2026.00719,FairVLM: Enhancing Fairness and Prompt Sensiti...,Não,Sem TL,FairVLM com backbone SAMed; FairVLM com backbo...,FairVLM,Glaucoma e alterações do nervo óptico avaliada...,Harvard-FairSeg; MosMedData+; QaTa-COV19,2 regiões de segmentação: optic cup e optic rim,Sexo; raça; etnia; idioma; o conjunto contém u...,Publico,10.000 imagens no Harvard-FairSeg,Imagens de fundo de olho por Scanning Laser Op...,Segmentação,Semantic-Retaining Counterfactual Prompting; D...,"Representação desigual por sexo, raça, etnia e...",Dice; Intersection over Union; equity-scaled D...,Mitigar simultaneamente viés demográfico e sen...
4,10.64898/2026.05.29.728755,Equitable Health Intelligence: An Open Benchma...,Não,População de ancestralidade europeia do TCGA (...,Deep Neural Network piramidal; autoencoder lin...,Existentes,Câncer: 33 tipos de câncer e 7 agrupamentos pa...,The Cancer Genome Atlas; Protein Expression; m...,2 classes em cada tarefa e limiar temporal: ev...,Ancestralidade genética/população: European Am...,Publico,Aproximadamente 11.000 pacientes do TCGA; 1.47...,"Dados tabulares ômicos: expressão proteica, ex...",Classificação binária de prognóstico,Mixture-Gap e Independent-Gap para detectar di...,Predomínio de ancestralidade europeia no TCGA;...,AUROC; Mixture-Gap; Independent-Gap; Transfer-...,Escassez de dados nos grupos sub-representados...
5,10.1038/s41746-026-02868-z,FairGen: Preference-Aligned Diffusion for Demo...,Não,Stable Diffusion v1-4,FairGen; Stable Diffusion v1-4; FairDiffusion;...,FairGen,"Doenças dermatológicas; COVID-19, edema, opaci...",Fitzpatrick17k; CheXpert; COVID-19 Image Data ...,"Pele: 5 classes (Allergic Contact Dermatitis, ...",Tom de pele; gênero; idade; raça na validação ...,Publico,17.606 imagens 

# **Export de Artigos de Revisao com Perguntas e Respostas**

In [23]:
df.columns

Index(['DOI', 'Query_A', 'Base', 'Nome_A', 'Criterio_Final', 'Status_Final',
       'Questao_1', 'Questao_18', 'Questao_2', 'Questao_3', 'Questao_4',
       'Questao_5', 'Questao_6', 'Questao_7', 'Questao_8', 'Questao_10',
       'Questao_11', 'Questao_12', 'Questao_13', 'Questao_14', 'Questao_15',
       'Questao_16', 'Questao_17'],
      dtype='object')

In [24]:
df_perg_respostas = df.merge(
    df_respostas,
    how="left",
    left_on=["DOI"],
    right_on=["DOI"],
    suffixes=("_A","_B")
)

In [25]:
df_perg_respostas.columns

Index(['DOI', 'Query_A', 'Base', 'Nome_A', 'Criterio_Final', 'Status_Final',
       'Questao_1_A', 'Questao_18_A', 'Questao_2_A', 'Questao_3_A',
       'Questao_4_A', 'Questao_5_A', 'Questao_6_A', 'Questao_7_A',
       'Questao_8_A', 'Questao_10_A', 'Questao_11_A', 'Questao_12_A',
       'Questao_13_A', 'Questao_14_A', 'Questao_15_A', 'Questao_16_A',
       'Questao_17_A', 'Questao_1_B', 'Questao_18_B', 'Questao_2_B',
       'Questao_3_B', 'Questao_4_B', 'Questao_5_B', 'Questao_6_B',
       'Questao_7_B', 'Questao_8_B', 'Questao_10_B', 'Questao_11_B',
       'Questao_12_B', 'Questao_13_B', 'Questao_14_B', 'Questao_15_B',
       'Questao_16_B', 'Questao_17_B'],
      dtype='object')

In [26]:
import pandas as pd
import re


# Considera strings vazias ou contendo apenas espaços como valores ausentes
df_perg_respostas= df_perg_respostas.replace(r"^\s*$", pd.NA, regex=True)

# Identifica todas as questões que terminam em _A ou _B
padrao = re.compile(r"^(Questao_\d+)_(A|B)$")

questoes = {
    padrao.match(coluna).group(1)
    for coluna in df_perg_respostas.columns
    if padrao.match(coluna)
}

# Para cada questão:
# 1. usa o valor de _A quando disponível;
# 2. caso contrário, usa o valor de _B;
# 3. cria a coluna sem o sufixo.
for questao in questoes:
    coluna_a = f"{questao}_A"
    coluna_b = f"{questao}_B"

    if coluna_a in df_perg_respostas.columns and coluna_b in df_perg_respostas.columns:
        df_perg_respostas[questao] = df_perg_respostas[coluna_a].combine_first(df_perg_respostas[coluna_b])
    elif coluna_a in df_perg_respostas.columns:
        df_perg_respostas[questao] = df_perg_respostas[coluna_a]
    elif coluna_b in df_perg_respostas.columns:
        df_perg_respostas[questao] = df_perg_respostas[coluna_b]

# Exclui todas as colunas de questões com os sufixos _A e _B
colunas_questoes_originais = [
    coluna for coluna in df_perg_respostas.columns
    if padrao.match(coluna)
]

df_perg_respostas= df_perg_respostas.drop(columns=colunas_questoes_originais)

# Exclui as demais colunas indicadas.
# errors="ignore" evita erro caso alguma delas não exista.
colunas_excluir = [
    "Status_A",
    "Codigo_A_B",
    "Query_A_B",
    "Base_A_B",
    "Artigo_B",
    "Nome_A_B",
    "Codigo_B",
    "Query_B",
    "Base_B",
    "Status_B",
    "Nome_B",
    "Nome_20",
]

df_perg_respostas= df_perg_respostas.drop(columns=colunas_excluir, errors="ignore")

# Renomeia as colunas principais
df_perg_respostas= df_perg_respostas.rename(
    columns={
        "Codigo_A_A": "Codigo",
        "Query_A_A": "Query",
        "Base_A_A": "Base",
        "Artigo_A": "Artigo",
        "Nome_A_A": "Nome",
    }
)

# Opcional: organiza as colunas Questao_* em ordem numérica
colunas_questoes = sorted(
    [coluna for coluna in df_perg_respostas.columns if re.fullmatch(r"Questao_\d+", coluna)],
    key=lambda coluna: int(coluna.split("_")[1]),
)

outras_colunas = [
    coluna for coluna in df_perg_respostas.columns
    if coluna not in colunas_questoes
]

df_perg_respostas= df_perg_respostas[outras_colunas + colunas_questoes]
df_perg_respostas.shape

(161, 23)

In [27]:
df_perg_respostas.columns

Index(['DOI', 'Query_A', 'Base', 'Nome_A', 'Criterio_Final', 'Status_Final',
       'Questao_1', 'Questao_2', 'Questao_3', 'Questao_4', 'Questao_5',
       'Questao_6', 'Questao_7', 'Questao_8', 'Questao_10', 'Questao_11',
       'Questao_12', 'Questao_13', 'Questao_14', 'Questao_15', 'Questao_16',
       'Questao_17', 'Questao_18'],
      dtype='object')

In [28]:
df_perg_respostas.columns  = [['DOI', 'Query_A', 'Base', 'Nome_A', 'Criterio_Final', 'Status_Final',
       'Questao_1', 'Questao_2', 'Questao_3', 'Questao_4', 'Questao_5',
       'Questao_6', 'Questao_7', 'Questao_8', 'Questao_10', 'Questao_11',
       'Questao_12', 'Questao_13', 'Questao_14', 'Questao_15', 'Questao_16',
       'Questao_17', 'Questao_18']]

In [29]:
df_perg_respostas.head()

,DOI,Query_A,Base,Nome_A,Criterio_Final,Status_Final,Questao_1,Questao_2,Questao_3,Questao_4,...,Questao_8,Questao_10,Questao_11,Questao_12,Questao_13,Questao_14,Questao_15,Questao_16,Questao_17,Questao_18
0,10.1007/978-981-97-7679-5_1,6.0,Scopus,CLASSIFICATION OF GOUGEROT SJOGREN SYNDROME BA...,Parecido com nosso objetivo,1.0,CLASSIFICATION OF GOUGEROT SJOGREN SYNDROME BA...,Sem TL,"U-Net (adaptado), FCN e VGG (encoder)",Existentes. A Y-Net é apenas uma extensão da U...,...,Sem atributos sensiveis,Publico e Privado,Não especificado,Imagens de ultrassonografia (US) das glândulas...,Classificação e segmentação,Não mencionado,Não mencionado,Dice coefficient\nIntersection over Union (IoU...,Variabilidade na qualidade e aquisição das ima...,Não
1,10.1016/j.jtos.2024.08.002,6.0,Scopus,DEEP LEARNING BASED ANALYSIS OF IN VIVO CONFOC...,NaN,1.0,Deep-learning based analysis of in-vivo confoc...,Não mencionado,SNP-Net,Novo,...,"Sim (idade, genero, condições sistemicas",Privados,Pacientes: 104.\r\nOlhos analisados: 160.\r\nD...,imagens de microscopia confocal in vivo (IVCM,Segmentação,Não mencionado,Representatividade Geográfica\r\nDistribuição ...,Métricas de Segmentação (Dice Similarity Coeff...,Qualidade e Variabilidade das Imagens\r\nTama...,Não
2,10.1038/s41598-023-42719-5,6.0,Scopus,APPLICATION OF SERUM SERS TECHNOLOGY COMBINED ...,NaN,1.0,Application of serum SERS technology combined ...,Não mencionado,AlexNet\r\nResNet (Residual Network)\r\nSqueez...,"AlexNet, ResNet, SqueezeNet e TCN são existen...",...,Sim(idade e genero),Privados,pSS: 27 espectros.\r\nDN: 30 espectros.\r\nHC:...,espectroscopia Raman aprimorada por superfície...,Classificação,Não mencionado,1. Viés de tamanho da amostra\r\n2. Viés demog...,\r\nAcurácia (Accuracy)\r\nSensibilidade (Sens...,1. Tamanho limitado da amostra\r\n2. Variabili...,Não
3,10.1016/j.bspc.2026.109615,6.0,Scopus,DIAGNOSTIC PERFORMANCE OF DEEP LEARNING MODELS...,Parecido com nosso objetivo,1.0,Diagnostic performance of deep learning models...,ImageNet,YOLO11n-cls\nYOLO11s-cls\nYOLO11m-cls\nYOLO11l...,Existentes,...,Sem atributos sensiveis,Privado,Não especificado,Ultrassonografia (US) das glândulas parótidas,Classificação,Não foi utilizada abordagem de fairness,Não especificado,Acurácia\nSensibilidade\nEspecificidade\nAUC,Dificuldade no diagnóstico precoce da Síndrome...,NaN
4,10.3390/diagnostics13142373,6.0,Scopus,DETECTION OF HYDROXYCHLOROQUINE RETINOPATHY VI...,NaN,1.0,Detection of Hydroxychloroquine Retinopathy vi...,ImageNet,- ResNet (Residual Network)\r\n- VGG (Visual ...,Existentes:\r\n- ResNet (Residual Network)\r\n...,...,Não,Privados,Não mencionado,imagens hiperespectrais oftalmoscópicas,classificação,Não mencionado,Viés de amostra: Falta de diversidade demográf...,\r\nAcurácia (Accuracy)\r\nPrecisão (Precision...,1. Tamanho limitado e diversidade dos dados.\r...,Não


# Step 2: Export Base Final

In [30]:
df_perg_respostas.to_csv(path + '3_Lista_Final_Artigos_Revisao_202607.csv', index=False)